In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path

#
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.GX_271.dim import DIM
from Instruments.WPI_Syr_pump.Pump import AL1000

#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.103", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to the experiment "Development") ---
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=35.6,
    y_offset=8.1,
    x_min=27,
    x_max=127,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=201,
    y_offset=39,
    x_min=157,
    x_max=247,
    y_min=2,
    y_max=292
) 

In [2]:
# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

syringe_dia = 4.606    # mm
rate = 0.5            # mL/min
volume = 1.2           # mL
direction = "INF"     # Infuse

pump.prepare(dia=syringe_dia, rate=rate, volume=volume, direction=direction)

Device is connected

--- Preparing pump (Phase 1) ---
Sent: @00*PHN1
Sent: @00*DIA4.606
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*VOL1.2
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRINF
Raw reply: 00S
Parsed reply: Acknowledged = True
--- Preparation complete ---



In [12]:
g.go_to_vial("rack2", 2)

(np.float64(201.0), np.float64(106.0))

In [10]:
g.move_z(44, allow_in_vial=True)

'Moved Z to 44. Result: Move Z: 2, Success'

In [14]:
g.send_command("Get XY Position")

'Get XY Position: 35.6, 8.1'

In [52]:
g.move_y(2)

[DEBUG] ensure_z_safe: current_z=100.0 is already >= target_z=100


'Moved Y to 2. Result: No valid response received.'

In [60]:
g.move_x(247)

[DEBUG] ensure_z_safe: current_z=100.0 is already >= target_z=100


'Moved X to 247. Result: No valid response received.'

In [21]:
g.go_into_vial("rack2", 1,)

[DEBUG] ensure_z_safe: current_z=95 is already >= target_z=93
[DEBUG] ensure_z_safe: current_z=95 is already >= target_z=45.0


(np.float64(201.0), np.float64(39.0), 1)

In [15]:
print(g.current_x)

247.0


In [10]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=93 is already >= target_z=45.0


(np.float64(201.0), np.float64(39.0))

In [11]:
print(g.current_module)

rack1
